# 📚 WeebCentral Manga Downloader

Download manga chapters from WeebCentral with full parallel support.

### Setup
1. Run **Cell 1** once to install dependencies + Clone Repository.
2. Run **Cell 2** — paste URL, pick chapters & format, everything downloads.
3. Optionally run **Cell 3** to zip & download to your PC.

### Output formats
| Option | Result |
|--------|--------|
| `pdf` | One PDF per chapter |
| `cbz` | CBZ archive with `ComicInfo.xml` (Kavita / Komga / CDisplayEx) |
| `images` | Raw image folder per chapter |
| `all` | PDF + CBZ + Images |

### Supported URL
```
https://weebcentral.com/series/SERIES_ID/manga-slug
```

In [1]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 1 — Install dependencies  (run once)          ║
# ╚══════════════════════════════════════════════════════╝
!pip install -q requests 'httpx[http2]' nest_asyncio beautifulsoup4 lxml Pillow fpdf2 tqdm
!pip install PyPDF2
print('✅ Dependencies ready.')
!git clone https://github.com/Yui007/weebcentral_downloader

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.8 MB/s eta 0:00:00
✅ Dependencies ready.
Cloning into 'weebcentral_downloader'...
remote: Enumerating objects: 213, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 213 (delta 25), reused 24 (delta 22), pack-reused 177 (from 1)
Receiving objects: 100% (213/213), 625.87 KiB | 9.48 MiB/s, done.
Resolving deltas: 100% (105/105), done.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 2 — Scrape, select & download                 ║
# ╚══════════════════════════════════════════════════════╝
import sys
import os
from PyPDF2 import PdfMerger
from pathlib import Path

sys.path.insert(0, '/content/weebcentral_downloader/colab')

from colab_scraper    import scrape_manga_info, scrape_chapter_list
from colab_downloader import parse_chapter_selection, download_chapters

def merge_pdfs_to_target_size(pdf_files, output_path, target_size_mb=200):
    """
    Merge PDF files into larger files up to target_size_mb.
    Returns list of created merged files.
    """
    target_bytes = target_size_mb * 1024 * 1024
    merged_files = []

    if not pdf_files:
        return merged_files

    current_batch = []
    current_size = 0

    for pdf_file in pdf_files:
        file_size = os.path.getsize(pdf_file)

        # If this file alone exceeds target size, create separate file for it
        if file_size > target_bytes:
            # Save current batch if exists
            if current_batch:
                merged_file = merge_pdf_batch(current_batch, output_path, len(merged_files) + 1)
                merged_files.append(merged_file)
                current_batch = []
                current_size = 0

            # Create single file for this large PDF
            single_merged = merge_pdf_batch([pdf_file], output_path, len(merged_files) + 1)
            merged_files.append(single_merged)
            continue

        # Check if adding this file would exceed target size
        if current_size + file_size > target_bytes and current_batch:
            # Save current batch and start new one
            merged_file = merge_pdf_batch(current_batch, output_path, len(merged_files) + 1)
            merged_files.append(merged_file)
            current_batch = [pdf_file]
            current_size = file_size
        else:
            # Add to current batch
            current_batch.append(pdf_file)
            current_size += file_size

    # Save the last batch
    if current_batch:
        merged_file = merge_pdf_batch(current_batch, output_path, len(merged_files) + 1)
        merged_files.append(merged_file)

    return merged_files

def merge_pdf_batch(pdf_list, output_dir, batch_num):
    """Merge a list of PDF files into one."""
    if not pdf_list:
        return None

    merger = PdfMerger()
    for pdf_file in pdf_list:
        merger.append(pdf_file)

    # Create output filename with batch number
    base_name = Path(pdf_list[0]).stem.split('_')[0]  # Get manga name from first file
    output_file = os.path.join(output_dir, f"{base_name}_batch_{batch_num}.pdf")

    # Ensure we don't overwrite existing files
    counter = 1
    while os.path.exists(output_file):
        output_file = os.path.join(output_dir, f"{base_name}_batch_{batch_num}_{counter}.pdf")
        counter += 1

    merger.write(output_file)
    merger.close()

    return output_file

def process_merged_pdfs(original_dir, merged_dir, target_size_mb=200):
    """Process all PDFs in original_dir and create merged versions in merged_dir."""
    # Create merged directory if it doesn't exist
    os.makedirs(merged_dir, exist_ok=True)

    # Find all PDF files in original directory
    pdf_files = sorted([os.path.join(original_dir, f) for f in os.listdir(original_dir)
                        if f.endswith('.pdf') and not f.startswith('merged_')])

    if not pdf_files:
        return []

    # Merge PDFs
    merged_files = merge_pdfs_to_target_size(pdf_files, merged_dir, target_size_mb)

    return merged_files

# ── 1. URL ───────────────────────────────────────────────────────────────────
SERIES_URL = input(
    '🌐 WeebCentral series URL\n'
    '   e.g. https://weebcentral.com/series/01JCZQE.../manga-slug\n'
    '   > '
).strip()

# ── 2. Scrape ────────────────────────────────────────────────────────────────
manga_info = scrape_manga_info(SERIES_URL)
chapters   = scrape_chapter_list(SERIES_URL)
total      = len(chapters)

# ── 3. Show chapter list ─────────────────────────────────────────────────────
print(f"\n{'='*62}")
print(f"  {'#':>4}   {'Chapter':<50}  Date")
print(f"{'='*62}")
for ch in chapters:
    num  = str(ch['index']).rjust(4)
    name = ch['title'][:50].ljust(50)
    print(f"  {num}   {name}  {ch['date']}")
print(f"{'='*62}")
print(f"  Total: {total} chapters\n")

# ── 4. Chapter selection ─────────────────────────────────────────────────────
print('📥 Selection formats:')
print('   all          → every chapter')
print('   single 5     → chapter 5 only')
print('   range 1-10   → chapters 1 through 10 (inclusive)')
print('   1,5,9,15     → specific chapters')
print()

while True:
    sel_input = input(f'🎯 Select chapters (1–{total}): ').strip()
    try:
        selected = parse_chapter_selection(sel_input, total)
        print(f'   ✅ {len(selected)} chapter(s) selected.')
        break
    except ValueError as e:
        print(f'   ❌ {e}\n   Try again.')

# ── 5. Output format ─────────────────────────────────────────────────────────
print()
print('📦 Output format:')
print('   pdf    → PDF file per chapter')
print('   cbz    → CBZ with ComicInfo.xml  (Kavita / Komga / CDisplayEx)')
print('   images → Raw image folder per chapter')
print('   all    → PDF + CBZ + Images')
print()

while True:
    fmt = input('🗂  Format [pdf / cbz / images / all]: ').strip().lower()
    if fmt in ('pdf', 'cbz', 'images', 'all'):
        break
    print('   ❌ Please enter pdf, cbz, images, or all.')

# ── 6. Download ──────────────────────────────────────────────────────────────
output_dir = download_chapters(
    manga_info       = manga_info,
    chapters         = chapters,
    selected_indices = selected,
    output_format    = fmt,
    output_dir       = '/content/weebcentral_downloader/colab/manga',
)

# ── 7. Merge PDFs if PDF format was requested ────────────────────────────────
if fmt in ('pdf', 'all'):
    print("\n🔄 Merging PDFs to target size (max 200MB per file)...")

    # Define directories
    original_pdf_dir = output_dir  # PDFs are saved in the main output directory
    merged_pdf_dir = os.path.join(output_dir, 'merged_pdfs')

    # Process and merge PDFs
    merged_files = process_merged_pdfs(original_pdf_dir, merged_pdf_dir, target_size_mb=200)

    if merged_files:
        print(f"✅ Created {len(merged_files)} merged PDF file(s) in '{merged_pdf_dir}':")
        for i, mf in enumerate(merged_files, 1):
            size_mb = os.path.getsize(mf) / (1024 * 1024)
            print(f"   {i}. {os.path.basename(mf)} ({size_mb:.2f} MB)")

        print(f"\n📁 Original PDFs (per chapter) are still available in: {original_pdf_dir}")
        print(f"📁 Merged PDFs are available in: {merged_pdf_dir}")
    else:
        print("⚠️  No PDF files found to merge.")

🌐 WeebCentral series URL
   e.g. https://weebcentral.com/series/01JCZQE.../manga-slug
   > https://weebcentral.com/series/01J76XY9E3JSAWKXW3Q36SQ7C6/Toukyou-Kushu
🔎 Fetching series page: https://weebcentral.com/series/01J76XY9E3JSAWKXW3Q36SQ7C6/Toukyou-Kushu

📖 Title    : Tokyo Ghoul
👤 Authors  : ISHIDA Sui
🏷  Tags     : Action, Drama, Psychological, Seinen, Supernatural
📌 Type     : Manga
📊 Status   : Complete
📅 Released : 2011
🖼  Cover    : https://temp.compsci88.com/cover/fallback/01J76XY9E3JSAWKXW3Q36SQ7C6.jpg

📝 Description:
Shy Ken Kaneki is thrilled to go on a date with the beautiful Rize. But it turns out that she’s only interested in his body - eating it, that is. When a morally questionable rescue transforms him into the first half-human half-Ghoul hybrid, Ken is drawn into the dark and violent world of Ghouls, whi...

🔎 Fetching chapter list: https://weebcentral.com/series/01J76XY9E3JSAWKXW3Q36SQ7C6/full-chapter-list

📚 Total chapters: 144
   Ch 1 (oldest) : Chapter 1 Last R

  Ch001:   0%|          | 0/47 [00:00<?, ?pg/s]

       🖼  30 pages — downloading in parallel...


  Ch002:   0%|          | 0/30 [00:00<?, ?pg/s]

       🖼  26 pages — downloading in parallel...


...

          → Tokyo Ghoul - Ch040 - Chapter 40 Last Read 2024-09-07T17_04_15.717343Z.pdf

[43/144] 📖  Chapter 43 Last Read 2024-09-07T17:04:15.717343Z  (2024-09-07T17:04:15.717343Z)
          → Tokyo Ghoul - Ch041 - Chapter 41 Last Read 2024-09-07T17_04_15.717343Z.pdf

[44/144] 📖  Chapter 44 Last Read 2024-09-07T17:04:15.717343Z  (2024-09-07T17:04:15.717343Z)
       🖼  18 pages — downloading in parallel...


  Ch043:   0%|          | 0/18 [00:00<?, ?pg/s]

       🖼  18 pages — downloading in parallel...


  Ch044:   0%|          | 0/18 [00:00<?, ?pg/s]

       ✅ 18/18 pages
       📄 Building PDF...


In [3]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 3 — (Optional) Zip & download to your PC      ║
# ╚══════════════════════════════════════════════════════╝
import re
import shutil
from google.colab import files
import os

try:
    # Определяем директории
    original_pdf_dir = output_dir  # Директория с исходными PDF-файлами
    merged_pdf_dir = os.path.join(output_dir, 'merged_pdfs')  # Директория со склеенными файлами

    # Проверяем, какой формат был выбран
    if fmt in ('pdf', 'all'):
        # Если есть склеенные файлы, архивируем только их
        if os.path.exists(merged_pdf_dir) and os.listdir(merged_pdf_dir):
            # Проверяем, есть ли в merged_pdf_dir PDF файлы
            merged_files = [f for f in os.listdir(merged_pdf_dir) if f.endswith('.pdf')]

            if merged_files:
                # Создаем безопасное имя для архива
                zip_stem = re.sub(r'[\\/:*?"<>|]', '_', manga_info['title']).strip() + '_merged_manga'
                zip_path = f'/content/weebcentral_downloader/colab/{zip_stem}'

                print(f'📦 Creating archive from merged PDFs: {merged_pdf_dir}')
                print(f'📁 Found {len(merged_files)} merged PDF file(s) to archive')

                # Создаем zip архив из папки merged_pdfs
                shutil.make_archive(zip_path, 'zip', merged_pdf_dir)

                # Полный путь к созданному архиву
                zip_file = f'{zip_path}.zip'

                if os.path.exists(zip_file):
                    # Получаем размер архива
                    size_mb = os.path.getsize(zip_file) / (1024 * 1024)
                    print(f'✅ Archive created: {zip_stem}.zip ({size_mb:.2f} MB)')
                    print(f'📥 Initiating download...')

                    # Скачиваем архив
                    files.download(zip_file)

                    print('✅ Download completed!')
                    print(f'\n📌 Note: Only merged PDF files (up to 200MB each) were archived.')
                    print(f'   Original chapter PDFs were kept at: {original_pdf_dir}')
                else:
                    print(f'❌ Archive file not found: {zip_file}')
            else:
                print(f'⚠️ No PDF files found in merged directory: {merged_pdf_dir}')
        else:
            print(f'⚠️ Merged PDFs directory not found or empty: {merged_pdf_dir}')
            print(f'📌 No merged PDF files to download.')
    else:
        # Для других форматов (cbz, images) архивируем всё содержимое output_dir
        zip_stem = re.sub(r'[\\/:*?"<>|]', '_', manga_info['title']).strip() + '_manga'
        zip_path = f'/content/weebcentral_downloader/colab/{zip_stem}'

        print(f'📦 Creating archive from: {output_dir}')

        # Проверяем, существует ли директория
        if not os.path.exists(output_dir):
            print(f'❌ Error: Directory {output_dir} does not exist!')
        else:
            # Создаем zip архив
            shutil.make_archive(zip_path, 'zip', output_dir)

            # Полный путь к созданному архиву
            zip_file = f'{zip_path}.zip'

            if os.path.exists(zip_file):
                # Получаем размер архива
                size_mb = os.path.getsize(zip_file) / (1024 * 1024)
                print(f'✅ Archive created: {zip_stem}.zip ({size_mb:.2f} MB)')
                print(f'📥 Initiating download...')

                # Скачиваем архив
                files.download(zip_file)

                print('✅ Download completed!')
            else:
                print(f'❌ Archive file not found: {zip_file}')

except Exception as e:
    print(f'❌ Error during archiving/download: {str(e)}')

📦 Creating archive from merged PDFs: /content/weebcentral_downloader/colab/manga/20th Century Boys/merged_pdfs
📁 Found 7 merged PDF file(s) to archive
✅ Archive created: 20th Century Boys_merged_manga.zip (1287.25 MB)
📥 Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download completed!

📌 Note: Only merged PDF files (up to 200MB each) were archived.
   Original chapter PDFs were kept at: /content/weebcentral_downloader/colab/manga/20th Century Boys
